[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_4_Q_learning_frozenlake.ipynb)

<div style="text-align:center">
    <h1>
        Q-Learning
    </h1>
</div>
<br>

<div style="text-align:center">
    <p>
        Q-Learning changes exactly one term in the SARSA update &mdash; and that one term changes what the agent is learning about. It is the most famous algorithm in reinforcement learning, and the direct ancestor of Deep Q-Networks.
    </p>
</div>

In [ ]:
# Setup: install Gymnasium and fetch the shared course helpers.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('utils_frozenlake.py'):
    !pip install -qq gymnasium==1.3.0 pygame seaborn
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils_frozenlake.py

from utils_frozenlake import (plot_values, plot_policy, plot_action_values,
                              plot_stats, test_agent, evaluate_policy, seed_everything)

## Import the necessary software libraries:

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## From SARSA to Q-Learning: one term, derived

**Recall SARSA's update.** After experiencing $(s, a, r, s')$ and choosing the next action $a'$:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[\, r + \gamma\, Q(s', a') - Q(s,a) \,\bigr]$$

The target samples the Bellman **expectation** equation of the policy being followed &mdash; $\varepsilon$-greedy, exploration included. SARSA answers: *"how good is this action if I keep behaving as I currently behave?"*

**The alternative question.** Usually we want something stronger: *"how good is this action if I behave optimally afterwards?"* The recursion answering it is the Bellman **optimality** equation &mdash; the same one value iteration used:

$$Q_*(s, a) = \mathbb{E}\bigl[\, r + \gamma\, \max_{a'} Q_*(s', a') \,\bigr]$$

Sampling *this* expectation instead gives the **Q-Learning** update:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[\, r + \gamma\, \max_{a'} Q(s', a') - Q(s,a) \,\bigr]$$

The only change: $Q(s', a')$ became $\max_{a'} Q(s', a')$. The target no longer cares which action the agent will *actually* take next &mdash; it always evaluates the *best* one.

**Why this is called off-policy.** Two policies are now involved:

| | Policy |
|---|---|
| **Behavior policy** &mdash; generates the experience | $\varepsilon$-greedy (keeps exploring) |
| **Target policy** &mdash; whose value is being learned | greedy (the max in the target) |

In SARSA the two coincide (**on-policy**); in Q-Learning they differ (**off-policy**). The practical consequence: Q-Learning estimates optimal values directly, even while behaving imperfectly. This decoupling is also what later allows learning from *replayed* or even *other agents'* experience &mdash; the foundation of Deep Q-Networks.

**The trade-off, honestly stated.** SARSA's estimates account for the agent's own exploration mistakes, so during training it favors "safer" routes; the classic illustration is Sutton &amp; Barto's cliff-walking grid, where SARSA walks away from the cliff edge and Q-Learning hugs it. On Frozen Lake the environment's own randomness dominates the exploration noise, so the two converge to similar answers &mdash; we will verify this with a head-to-head comparison at the end.

**Both decay schedules carry over.** As derived in the SARSA notebook, $\varepsilon$ decays so exploitation can take over, and $\alpha$ decays so noisy samples average out instead of endlessly jostling the table.

**Same common mistake as before:** at a terminal $s'$ the target is just $r$ &mdash; the `(not done)` mask on the bootstrap term is still essential.

## Exploration is still required

Off-policy does not mean exploration-free: the max in the target fixes what the agent learns *about*, but the agent must still *visit* state&ndash;action pairs to update them. We reuse the $\varepsilon$-greedy machinery.

Before the algorithm, one question must be answered: **while learning, which actions should the agent take?**

Acting greedily (always the best-looking action) fails immediately here. The Q table starts at all zeros; a greedy agent breaks ties arbitrarily, repeats the same path, and &mdash; because the reward is sparse &mdash; may *never* stumble on the gift. An estimate that is never tested is never corrected. This is the **exploration&ndash;exploitation trade-off**: to learn, the agent must sometimes try actions its current table considers inferior.

The simplest fix is the **$\varepsilon$-greedy** policy:

- with probability $\varepsilon$: pick an action uniformly at random (**explore**),
- with probability $1-\varepsilon$: pick the action with the highest $Q(s,a)$ (**exploit**).

We start with $\varepsilon = 1$ (pure exploration, since the table knows nothing) and decay it toward a small floor as the table becomes trustworthy. The floor is kept above zero so that no action is ever permanently abandoned.

**App-builder note.** $\varepsilon$ is a natural slider in a teaching tool: drag it and watch the agent's behavior shift from random wandering to confident routing. The decay schedule is the second control worth exposing.

In [ ]:
def epsilon_greedy(Q, state, epsilon):
    """Explore with probability epsilon; otherwise act greedily w.r.t. Q."""
    if np.random.random() < epsilon:
        return np.random.randint(Q.shape[1])
    return int(np.argmax(Q[state]))


def linear_schedule(episode, total_episodes, start, floor):
    """Linear decay from `start` to `floor` over the first 90% of training.
    Used for BOTH epsilon (exploration) and alpha (learning rate)."""
    return max(floor, start * (1 - episode / (0.9 * total_episodes)))

## The Q-Learning algorithm

Compare the loop with SARSA's: the next action is chosen *after* the update, and the target uses `np.max` &mdash; those are the only structural differences.

<div style="text-align:center">
    Adapted from Barto & Sutton: "Reinforcement Learning: An Introduction", Ch. 6.
</div>

In [ ]:
def q_learning(env, episodes, gamma=0.99):
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    returns = []

    for episode in range(episodes):
        epsilon = linear_schedule(episode, episodes, start=1.0, floor=0.05)
        alpha   = linear_schedule(episode, episodes, start=0.5, floor=0.01)
        state, _ = env.reset()
        done, total_reward = False, 0.0

        while not done:
            action = epsilon_greedy(Q, state, epsilon)      # behavior policy: explores
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Target policy: the max evaluates the GREEDY action at s',
            # regardless of what the agent will actually do next (off-policy).
            target = reward + gamma * np.max(Q[next_state]) * (not done)
            Q[state, action] += alpha * (target - Q[state, action])

            state = next_state
            total_reward += reward

        returns.append(total_reward)
    return Q, returns

## First test: firm ice

A practical detail before training: we create the training environment **without** `render_mode`. Rendering draws a picture on every step, which is useful for watching but roughly a hundred times slower than pure computation. We train blind and fast, and create a second, rendering environment only when we want to watch the finished agent.

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
seed_everything(env, seed=42)

Q, returns = q_learning(env, episodes=2000)
plot_stats({"Returns (success rate, smoothed)": returns})

In [ ]:
env_render = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array")
env_render.reset(seed=42)
frame = env_render.render()

plot_values(Q.max(axis=1), frame=frame, env=env_render)
plot_policy(Q, frame=frame, env=env_render)

In [ ]:
print(f"Greedy-policy success rate on firm ice: "
      f"{evaluate_policy(env, lambda s: int(np.argmax(Q[s]))):.1%}")

## The real test: slippery ice

In [ ]:
env_slip = gym.make("FrozenLake-v1", is_slippery=True)
seed_everything(env_slip, seed=42)

np.random.seed(0)
Q_slip, returns_slip = q_learning(env_slip, episodes=10000)
plot_stats({"Returns (success rate, smoothed)": returns_slip})

In [ ]:
env_slip_render = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
env_slip_render.reset(seed=42)

plot_values(Q_slip.max(axis=1), env=env_slip_render)
plot_policy(Q_slip, env=env_slip_render)
plot_action_values(Q_slip, env=env_slip_render)

In [ ]:
print(f"Q-Learning greedy policy on slippery ice: "
      f"{evaluate_policy(env_slip, lambda s: int(np.argmax(Q_slip[s]))):.1%}")

In [ ]:
test_agent(env_slip_render, lambda s: int(np.argmax(Q_slip[s])), episodes=3)

## Head-to-head: SARSA vs Q-Learning

Same environment, same hyperparameters, same number of episodes &mdash; only the update rule differs. We redefine SARSA compactly here so the comparison is self-contained.

In [ ]:
def sarsa(env, episodes, gamma=0.99):
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    returns = []
    for episode in range(episodes):
        epsilon = linear_schedule(episode, episodes, start=1.0, floor=0.05)
        alpha   = linear_schedule(episode, episodes, start=0.5, floor=0.01)
        state, _ = env.reset()
        action = epsilon_greedy(Q, state, epsilon)
        done, total_reward = False, 0.0
        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_action = epsilon_greedy(Q, next_state, epsilon)
            # On-policy target: the action the agent will ACTUALLY take next.
            target = reward + gamma * Q[next_state, next_action] * (not done)
            Q[state, action] += alpha * (target - Q[state, action])
            state, action = next_state, next_action
            total_reward += reward
        returns.append(total_reward)
    return Q, returns


np.random.seed(0)     # reseed so both algorithms train from identical randomness
Q_sarsa, returns_sarsa = sarsa(env_slip, episodes=10000)

def smooth(x, w=250):
    # Wide moving average: per-episode returns are 0/1, so heavy smoothing
    # is needed for the success-rate trend to be visible.
    return np.convolve(x, np.ones(w) / w, mode='valid')

plt.figure(figsize=(9, 4))
plt.plot(smooth(returns_slip), color="#7B2FBE", label="Q-Learning")
plt.plot(smooth(returns_sarsa), color="#E8912D", label="SARSA")
plt.xlabel("Episode"); plt.ylabel("Success rate (moving average)")
plt.title("Learning curves on slippery ice")
plt.legend(); plt.tight_layout(); plt.show()

sr_q = evaluate_policy(env_slip, lambda s: int(np.argmax(Q_slip[s])))
sr_s = evaluate_policy(env_slip, lambda s: int(np.argmax(Q_sarsa[s])))
print(f"Final greedy success rate - Q-Learning: {sr_q:.1%}   SARSA: {sr_s:.1%}   (value iteration benchmark: ~74%)")

Both algorithms recover essentially the optimal success rate from experience alone. As predicted, on Frozen Lake the on-/off-policy distinction barely moves the final numbers &mdash; the lake's own randomness swamps the exploration noise that separates them. The distinction becomes decisive in environments where exploration itself is dangerous (cliff-walking) and in everything that follows in this course: Deep Q-Networks are possible precisely *because* Q-Learning is off-policy and can learn from replayed old experience.

In [ ]:
env.close(); env_render.close(); env_slip.close(); env_slip_render.close()

## Summary

| | SARSA | Q-Learning |
|---|---|---|
| Target | $r + \gamma\, Q(s', a')$ | $r + \gamma\, \max_{a'} Q(s', a')$ |
| Bellman equation sampled | expectation (of the followed policy) | optimality |
| Type | on-policy | off-policy |
| Learns the value of | the exploring policy it follows | the greedy policy, while exploring |
| Character during training | cautious near dangers | direct, optimistic |
| Leads to | Expected SARSA, actor-critic | DQN and successors |

The whole tabular story is now complete: **dynamic programming** (model available), **temporal-difference learning** (experience only). The remaining limitation is the table itself &mdash; 16 states fit in a table, a camera image does not. Replacing the table with a neural network is the next chapter of the course.

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 6: Temporal-Difference Learning](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)

[[2] Watkins & Dayan (1992). Q-Learning. Machine Learning, 8, 279-292.](https://link.springer.com/article/10.1007/BF00992698)